# 02 · Your first race-order prediction

**Goal:** turn a hypothesis into a working prediction, save it, and interpret its errors.

Our first hypothesis is: **qualifying order is a reasonable starting point for predicting race order.** This benchmark will give us something concrete for future trained models to improve on.
We are reconstructing the 2024 Bahrain weekend from historical data. This is not evidence of live or historical point-in-time performance.

## Setup

Complete the data checks and exercise in notebook 01 before moving on.
Kernel: **WeAreChecking (.venv)**. Race results are kept out of the input table.

In [ ]:
import importlib
from IPython.display import display
from wearechecking.paths import project_root
from wearechecking.data import latest_snapshot
from wearechecking.features import qualifying_inputs
from wearechecking.evaluation import save_prediction, evaluate_saved
from wearechecking.learning import check_baseline
from wearechecking import student

ROOT = project_root()
snapshot = latest_snapshot(2024, 1, root=ROOT)
CUTOFF = '2024-03-02T12:00:00Z'

## Steps

### 1. Choose the prediction time

The cutoff is after qualifying ends and before the race starts.
Because we downloaded the data later, it may contain subsequent qualifying corrections. The `reconstructed` label records that limitation.
Qualifying participants are provisional race entries for this exercise. Penalties, substitutions, and DNS handling come later.

In [ ]:
inputs = qualifying_inputs(snapshot, CUTOFF, mode='reconstructed')
features = inputs.frame
print('Cutoff:', inputs.metadata['cutoff_utc'])
print('Input columns:', features.columns.tolist())
display(features.head(8))

### 2. Implement the hypothesis: qualifying_baseline

Edit the second function in `src/wearechecking/student.py`.
Input: driver_id, driver, team, qualifying_position.
Output: **driver_id, predicted_position**.

Hints:

1. A smaller qualifying_position means a better position.
2. Select and copy the two columns you need.
3. Sort the rows and give the predicted-position column its required name.
4. Use the number of drivers in the input rather than hardcoding 20.

**Think first:** Why would using Position from the R session be data leakage?

In [ ]:
importlib.reload(student)
prediction = student.qualifying_baseline(features.copy())
print(check_baseline(prediction, features))
if prediction is not None:
    display(prediction.sort_values('predicted_position').head(10))

### 3. Save the prediction

Create a prediction record before comparing it with the actual race result.
Each run creates a new record and preserves earlier predictions.

In [ ]:
prediction_folder = None
if prediction is None:
    print('PENDING: implement the prediction function first.')
else:
    check_baseline(prediction, features)
    prediction_folder = save_prediction(prediction, inputs, root=ROOT)
    print('Saved:', prediction_folder.name)

### 4. Open the actual result

**MAE:** the mean absolute difference between predicted and actual positions. Lower is better.
**Spearman:** the correlation between the two rankings; +1 means identical order and -1 means reversed order.

One historical race does not establish general performance. Later, we will evaluate multiple races in chronological order.
Classified drivers who did not finish retain their official positions, and the status column is shown. Missing or different entries require explicit handling.

In [ ]:
if prediction_folder is None:
    print('PENDING: no saved prediction yet; the result table has not been opened.')
else:
    comparison, metrics = evaluate_saved(prediction_folder, snapshot)
    display(comparison.sort_values('absolute_error', ascending=False).head(10))
    print('Position MAE:', round(metrics['position_mae'], 3))
    print('Spearman:', round(metrics['spearman'], 3))

## Checks

- There is one prediction per input driver, ranked 1..N.
- Race results were opened only after the prediction was saved.
- The check reports **PASSED**, and I can read the scores from my own output.
- I can explain why a score from one race cannot be generalized.

## Next steps

Complete `docs/DECISION_NOTE.md` using your own output.
Choose one candidate improvement: team history, driver history, or FP pace. We will first discuss how to evaluate it fairly.